# 2.2_huim — HUIM-inspired Stage 1 relevance model and catalogue shortlist

**Status of this notebook: implementation prepared, not yet executed.** No metric, training curve,
export, persistence or reload result exists until the cells below have been run in Colab. Cells that
print results only report what actually ran in that session; a prepared check is not evidence that it passed.

Pipeline position: *eligible article catalogue → **Stage 1 personalised shortlist (≤100)** → Stage 2
business reranker → final top 5*. This notebook implements Stage 1 only (no reranker, no ensemble).

Run order: `smoke` first (minutes, labelled smoke), then `development`, then `full` only after reading the
preflight storage/runtime estimates. Load-only: set `TRAIN_MODEL = False` and `LOAD_RUN_DIR`.

## 1. Business hypothesis and experiment overview

| Item | Value |
| --- | --- |
| Experiment ID | `2.2_huim` (neural track) |
| Owner | set `OWNER` below |
| Model | Compact HUIM-inspired click model: shared article tower, **candidate-conditioned attention over fixed-history items** (no recurrence), separate request/pair feature branch, concatenation or gated fusion, MLP head, BCE on logits |
| Business objective (Interim Report §1.1, §1.2.3) | Meaningful engagement and reader retention via relevant recommendations; Stage 1 protects **personal relevance through shortlist membership**; Stage 2 applies continuity, freshness and diversity within the shortlist |
| Hypothesis | Readers have several interests; weighting each past click by its similarity to the *candidate* (rather than averaging the whole history) should rank relevant articles higher on recorded slates and recover more known clicks in the catalogue top 100 |
| Evidence that would support it | Higher validation mean impression AUC (primary) / nDCG@5 than the **uniform-pooling control** at the same budget; higher known-click Recall@100 on the catalogue; attention not collapsing to uniform |
| Focused checks | (a) candidate attention vs uniform pooling; (b) concatenation vs gated fusion; (c) robustness when text/image vectors are hidden (most of the test catalogue has none) |

**Distinction from the parallel LSTM (2.1).** This model has no recurrence or positional encoding: the history is a
*set* of events, each carrying its own article, read-time bin, scroll bin and recency bin. Permuting aligned events
cannot change a score (unit-tested), while recency still influences attention because it stays attached to each event.

**What is adapted from HUIM (Xue et al.; BlackPearl code at commit `0d75efb`) and what is not**

| Aspect | HUIM paper | Released code | This adaptation |
| --- | --- | --- | --- |
| History pooling | DIN target attention, MLP([e_j, c, e_j−c, e_j⊙c]), softmax removed | FuxiCTR `DIN_Attention`, `din_use_softmax: false`, MLP [512,256] | Same input form, MLP [32,16]; **stable masked softmax by default**; the unnormalised masked sum (+LayerNorm, our addition) is an optional comparison; uniform pooling is a control |
| Item representation | category/subcat/sentiment/type embeddings + PCA-64 frozen text & image vectors (two attention groups) | same; PCA fitted on whole catalogue | One shared article tower: metadata embeddings + *separate* trained linear projections of frozen text (768-d) and image (1024-d) vectors, with missing-vector flags; no PCA; one attention group |
| History events | read-time and scroll bins | same, last 200 items | read-time/scroll quantile bins fitted on train history + fixed recency bins; last 50 fixed-history items |
| "Instant interest" | binned popularity/session/impression-level features (exposure counts, slate size, in-slate ranks…) | ~300 such features | **Not implemented**: these depend on the recorded slate, other users' test-period logs, or future totals. Nothing here is called "instant interest" |
| Fusion | BN(concat) → MLP, or gated (Eq. 11) | plain MLP (`CusDIN`); gated variant commented out | LayerNorm(concat[match, features]) → MLP; optional sigmoid gate (Eq. 11 style) |
| Loss | per-impression shifted logloss (stated not used for HUIM) | BCE | BCE on logits |
| History augmentation | val/test histories extended with later behaviour clicks | same | **No augmentation**: shared fixed histories only, events strictly before each split's first request |
| Leakage | paper reports 0.873 AUC with leaky features, 0.822 without | — | Only request-time-available inputs (audit table in §4) |

Course links: embeddings (metadata/article), attention (target attention), feature interactions
([h, c, h−c, h⊙c], preference crosses), dense networks (MLP head), regularisation (dropout, weight decay, modality
dropout, early stopping). The HUIM-style fusion as a whole is our adaptation, not an architecture taught in full.

**Evaluation pools (kept separate everywhere).** *Recorded slates*: the logged shown/clicked candidates; labelled;
mean impression AUC (primary) and nDCG@5 (secondary). *Catalogue*: every time-eligible article; outcomes of unshown
articles are unknown, so only known-click Recall@100 and coverage diagnostics are reported (no catalogue AUC/BCE).
Validation is used repeatedly for selection, so its scores are **development results**. Test has no labels.

## 2. Setup and run configuration

In [ ]:
# ---- Run configuration (edit here only) -------------------------------------------------
from pathlib import Path

RUN_MODE = "smoke"        # "smoke" (labelled smoke run) | "development" | "full"
TRAIN_MODEL = True        # False: load a saved bundle from LOAD_RUN_DIR and re-export without training
RESUME = True             # resume training/exports from the run's recovery state if present
SEED = 42
OWNER = "Winston Ho Rui Wei"

# Inputs (may be read-only). Use a shortcut path if the shared folder is not under MyDrive.
DATA_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder")
PATH_OVERRIDES = {}       # e.g. {"train_val_text_embeddings": "embeddings/train_val/text_embeddings.parquet"}

# Writable persistent storage for run bundles (separate from DATA_ROOT).
OUTPUT_ROOT = Path("/content/drive/MyDrive/BT4222 Experiment Runs/2.2_huim")
LOCAL_ROOT = Path("/content/huim_runs")    # fast session-local working copy. Export resume state lives here:
                                           # for multi-hour full exports set a Drive folder to survive VM loss.
RUN_ID = None             # None -> new "YYYYMMDD_HHMM_seed42_<mode>"; set an existing ID to resume it
LOAD_RUN_DIR = None       # TRAIN_MODEL=False: persistent run folder to load, e.g. OUTPUT_ROOT / "<run_id>"

# Code (private fork; authenticated with a Colab secret holding a GitHub token)
REPO_URL = "https://github.com/KoalaLyfestyle/BT4222-Final-Project.git"
REPO_BRANCH = "claude/focused-shannon-s4lmma"   # switch to "main" once merged
EXPECTED_COMMIT = None    # optional full SHA to pin the helper code
REPO_DIR = Path("/content/BT4222-Final-Project")
GITHUB_TOKEN_SECRET = "GITHUB_TOKEN"
ALLOW_DIRTY_CHECKOUT = False

# Optional, off by default (each costs extra training time)
RUN_INTERRUPTION_CHECK = False   # bounded resume check (~2 x 40 steps on a training subset)
RUN_CONTROLS = False             # uniform-pooling and gated-fusion comparisons (recorded slates only)
RUN_TUNING = False               # bounded Optuna search on validation AUC (installs optuna)
TUNING_TRIALS, TUNING_MAX_EPOCHS, TUNING_TRAIN_IMPRESSIONS = 8, 2, 50_000

CONFIG_OVERRIDES = {}     # e.g. {"max_epochs": 3, "batch_size": 2048}

In [ ]:
# ---- Mount Drive and probe storage -------------------------------------------------------
import os, time, uuid
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Colab: set DATA_ROOT/OUTPUT_ROOT to local paths.")

assert DATA_ROOT.is_dir(), f"DATA_ROOT not found: {DATA_ROOT}"
for sub in ("feature_engineering", "train_val", "test"):
    assert (DATA_ROOT / sub).is_dir(), f"{DATA_ROOT} lacks {sub}/ — is this the shared data folder?"

def write_probe(root: Path) -> dict:
    """Create, read back and delete a small file. The shared input folder is never probed."""
    root.mkdir(parents=True, exist_ok=True)
    probe = root / f".write_probe_{uuid.uuid4().hex}"
    payload = os.urandom(1024)
    probe.write_bytes(payload)
    ok = probe.read_bytes() == payload
    probe.unlink()
    return {"root": str(root), "writable": ok}

STORAGE_PROBES = {"output_root": write_probe(OUTPUT_ROOT), "local_root": write_probe(LOCAL_ROOT)}
print(STORAGE_PROBES)
assert all(p["writable"] for p in STORAGE_PROBES.values())

In [ ]:
# ---- Dependency check BEFORE importing helper modules (no blanket upgrades) -------------
import importlib, subprocess, sys
REQUIRED = {"numpy": "1.24", "pandas": "2.0", "pyarrow": "14.0", "torch": "2.3", "matplotlib": "3.7"}
def vtuple(v): return tuple(int(x) for x in v.split("+")[0].split(".")[:2] if x.isdigit())
problems = {}
for name, minimum in REQUIRED.items():
    try:
        mod = importlib.import_module(name)
        if vtuple(mod.__version__) < vtuple(minimum):
            problems[name] = f"{mod.__version__} < {minimum}"
        print(f"{name:10s} {mod.__version__}")
    except ImportError as error:
        problems[name] = f"missing ({error})"
if RUN_TUNING:
    try:
        import optuna
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optuna>=3.6,<5"], check=True)
        import optuna
    print("optuna", optuna.__version__)
assert not problems, f"Dependency problems (install only these packages): {problems}"

In [ ]:
# ---- Code checkout: authenticated fetch of the private fork, revision recorded --------------
import base64, shutil
def _token():
    try:
        from google.colab import userdata
        return userdata.get(GITHUB_TOKEN_SECRET)
    except Exception:
        return os.environ.get(GITHUB_TOKEN_SECRET)
TOKEN = _token()

def git(*args, cwd=REPO_DIR, auth=False, check=True):
    cmd = ["git"]
    if auth and TOKEN:  # header only for this call; the token is not written to .git/config or printed
        basic = base64.b64encode(f"x-access-token:{TOKEN}".encode()).decode()
        cmd += ["-c", f"http.https://github.com/.extraheader=AUTHORIZATION: basic {basic}"]
    result = subprocess.run(cmd + list(args), cwd=cwd, capture_output=True, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed: {result.stderr.replace(TOKEN or '<none>', '***')[:500]}")
    return result.stdout.strip()

if not (REPO_DIR / ".git").is_dir():
    git("clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR), cwd="/content", auth=True)
else:
    # Existing clone: never trust it blindly — fetch, refuse local edits, fast-forward.
    git("fetch", "origin", REPO_BRANCH, auth=True)
    dirty = git("status", "--porcelain")
    if dirty and not ALLOW_DIRTY_CHECKOUT:
        raise RuntimeError(f"Existing clone has local changes:\n{dirty}")
    git("checkout", REPO_BRANCH)
    git("merge", "--ff-only", f"origin/{REPO_BRANCH}")
if EXPECTED_COMMIT:
    git("checkout", EXPECTED_COMMIT)
CODE_REVISION = {"commit": git("rev-parse", "HEAD"), "branch": REPO_BRANCH,
                 "uncommitted_changes": bool(git("status", "--porcelain")),
                 "remote_head": git("rev-parse", f"origin/{REPO_BRANCH}", check=False)}
if EXPECTED_COMMIT:
    assert CODE_REVISION["commit"] == EXPECTED_COMMIT
print(CODE_REVISION)

In [ ]:
# ---- Import helpers from THIS checkout and verify consistency ------------------------------
import hashlib
EXP_DIR = REPO_DIR / "Experiments" / "2.2_huim"
for name in [m for m in sys.modules if m == "huim" or m.startswith("huim.")]:
    del sys.modules[name]          # drop stale modules from an earlier run of this kernel
if str(EXP_DIR) not in sys.path:
    sys.path.insert(0, str(EXP_DIR))
import huim
HELPER_HASHES = {}
for name in huim.HELPER_MODULES:
    module = importlib.import_module(f"huim.{name}")
    path = Path(module.__file__).resolve()
    assert path == (EXP_DIR / "huim" / f"{name}.py").resolve(), f"huim.{name} imported from {path}"
    HELPER_HASHES[name] = hashlib.sha256(path.read_bytes()).hexdigest()
tracked_changes = git("status", "--porcelain", "--", "Experiments/2.2_huim")
assert not tracked_changes or ALLOW_DIRTY_CHECKOUT, f"helper files differ from the commit:\n{tracked_changes}"

import json, numpy as np, pandas as pd, torch
from huim import pipeline as P, catalogue as C, features as F, data as D
from huim.io_utils import RunLogger, atomic_write_json, environment_info, read_json, dumps
from huim.model import parameter_counts

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    print("WARNING: no GPU; training and catalogue scoring will be very slow.")
SPLITS = D.SPLITS

CFG = P.make_config(RUN_MODE, seed=SEED, **CONFIG_OVERRIDES)
if not TRAIN_MODEL:
    assert LOAD_RUN_DIR is not None and P.run_is_complete(Path(LOAD_RUN_DIR)), \
        "LOAD_RUN_DIR must be a persisted run with a valid completion marker (written after its reload check passed)"
    saved_cfg = read_json(Path(LOAD_RUN_DIR) / "run_config.json")["config"]
    for key in ("history_len", "d_model", "attention_mode", "attention_hidden", "fusion", "feature_hidden",
                "head_hidden", "catalogue_policy", "vector_source_order", "text_dim", "image_dim"):
        CFG[key] = saved_cfg[key]   # reconstruction settings come from the saved bundle
RUN_ID = RUN_ID or f"{time.strftime('%Y%m%d_%H%M')}_seed{SEED}_{RUN_MODE}" + ("" if TRAIN_MODEL else "_reload")
RUN_DIR = LOCAL_ROOT / RUN_ID
PERSIST_DIR = OUTPUT_ROOT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
log = RunLogger(RUN_DIR)
log(f"run {RUN_ID} | mode {RUN_MODE} | device {DEVICE} | commit {CODE_REVISION['commit'][:10]}")
if P.run_is_complete(PERSIST_DIR):
    log(f"NOTE: {PERSIST_DIR} already has a valid completion marker; choose a new RUN_ID to rerun.")
print(json.dumps({k: v for k, v in CFG.items()}, indent=1, default=str)[:3000])

## 3. Load and check shared data

Loads behaviours (`train`, `validation`, `test`), fixed histories, the declared article catalogues, both vector
sources, and **only the keys and labels** of the shared exploded feature tables (training pairs = recorded slates;
no re-explosion, no catalogue negatives). Checks: composite-key uniqueness, label agreement with
`article_ids_clicked`, history alignment and the temporal cutoff, vector decoding (dimensions, finite values, flags,
overlap agreement), coverage, exact eligible-pair counts and storage estimates, and agreement of our request-time
features with the shared Step 3 features on the recorded pairs.

In [ ]:
with log.stage("prepare inputs"):
    PRE_LOADED = None
    if not TRAIN_MODEL:
        PRE_LOADED = F.Preprocessor.from_json(read_json(Path(LOAD_RUN_DIR) / "model" / "preprocessing.json"))
    ws = P.prepare_inputs(CFG, DATA_ROOT, pre=PRE_LOADED, path_overrides=PATH_OVERRIDES, log=log)
    INPUT_MANIFEST = D.input_manifest(ws.paths)
atomic_write_json(RUN_DIR / "input_manifest.json", INPUT_MANIFEST)
summary = pd.DataFrame({s: {"requests": len(ws.behaviours[s]), "users": ws.behaviours[s].user_id.nunique(),
                            "recorded_pairs": len(ws.pairs[s]),
                            "clicked_pairs": int(np.nansum(ws.pairs[s].clicked)) if s != "test" else "unknown (no labels)",
                            "first_request": ws.behaviours[s].impression_time.min(),
                            "last_request": ws.behaviours[s].impression_time.max(),
                            "history_events_kept": ws.reports["history"][s]["events_kept"],
                            "catalogue_articles": ws.ctx[s].report["catalogue_articles"]} for s in SPLITS})
display(summary)
print(dumps(ws.reports["catalogues"]))

In [ ]:
with log.stage("preflight"):
    PREFLIGHT = P.preflight(ws, CFG, [LOCAL_ROOT, OUTPUT_ROOT])
atomic_write_json(RUN_DIR / "preflight.json", PREFLIGHT)
feas = pd.DataFrame({s: PREFLIGHT["eligibility"][s]["feasibility"] for s in SPLITS}).T
display(feas[["requests", "eligible_pairs", "empty_pools", "pools_below_shortlist_size",
              "full_scores_csv_bytes_estimate"]])
print(f"All splits: {PREFLIGHT['all_splits']['eligible_pairs']:,} eligible pairs; "
      f"full score CSVs ≈ {PREFLIGHT['all_splits']['full_scores_csv_gib_estimate']:,.1f} GiB (formula estimate)")
display(pd.DataFrame(PREFLIGHT["coverage"]).T)
print("Recorded candidates not eligible in the catalogue:", PREFLIGHT["recorded_ineligible"])
print("Free bytes:", PREFLIGHT["disk_free_bytes"])
print("Consistency with shared Step 3 features:")
print(dumps(PREFLIGHT["consistency_with_shared_features"]))
for kind in ("text", "image"):
    for source, rep in PREFLIGHT["vector_decoding"][kind].items():
        print(kind, source, {k: rep.get(k) for k in ("rows", "dimension", "usable", "null_vectors", "wrong_length",
                                                    "non_finite", "overlap_max_abs_diff")})

## 4. Prepare model inputs

All inputs are rebuilt from source tables for *any* (request, article) pair, so training, recorded-slate scoring and
catalogue scoring share one feature path. Preprocessing is fitted on train only: Step 3 vocabularies (reused),
subcategory/sentiment vocabularies and body-length scaling (train-referenced articles), engagement quantile bins
(train history events), dense standardisation (train requests and train recorded pairs). Flags stay 0/1.

In [ ]:
audit = pd.DataFrame(F.FEATURE_AUDIT, columns=["feature", "scope", "status", "reason"])
with pd.option_context("display.max_colwidth", 200):
    display(audit)
if TRAIN_MODEL and "request_scaler" not in ws.pre.state:
    with log.stage("fit dense scalers on train"):
        F.fit_dense_scalers(ws.pre, ws.ctx["train"], ws.art)
FEATURE_ORDER = {"request_dense": F.REQUEST_DENSE_COLUMNS(ws.pre), "pair_dense": F.PAIR_DENSE_COLUMNS,
                 "article_inputs": ["category", "article_type", "sentiment_label", "subcategory (bag)", "topics (bag)",
                                    "premium", "premium_missing", "body_z", "body_missing", "topic_count_z",
                                    "has_text", "has_image", "known_article", "text_vector(768, frozen)",
                                    "image_vector(1024, frozen)"],
                 "history_event_inputs": ["article (shared tower)", "read_time_bin", "scroll_bin", "recency_bin"]}
MISSING_POLICY = {
    "article_not_in_lookup": "index 0 -> learned unknown-article representation (known=0)",
    "missing_text_or_image_vector": "projection contributes 0; has_text/has_image=0; article kept",
    "missing_published_time": "age feature 0 + article_age_missing=1; never catalogue-eligible",
    "future_published_recorded_candidate": "age clipped at 0 (training/diagnostics only; ineligible in catalogue)",
    "missing_premium/subscriber": "access_missing=1",
    "missing_read_time/scroll": "bin 0 (missing) per event; request aggregates have *_missing flags",
    "empty_history": "masked attention returns a learned empty-history vector; history_empty=1",
    "unknown_category/type/topic": "embedding 'unknown' index; cross preferences use all catalogue values",
}
mcfg_preview = P.model_config(CFG, ws.pre, ws.art)
print(json.dumps(mcfg_preview.to_json(), indent=1))
print({s: {k: tuple(v.shape) for k, v in ws.ctx[s].tensors.items() if k.startswith(("hist_", "req_dense"))} for s in SPLITS})

## 5. Define and train / load model

BCE on logits over the supplied recorded pairs; AdamW (no decay on biases/norms/embeddings), gradient clipping,
mixed precision on GPU, `ReduceLROnPlateau` on validation AUC, early stopping (patience 1), checkpoint selection by
**validation mean impression AUC** (ties → earlier epoch). Defaults are starting hypotheses, not tuned optima.
Recovery: `recovery/last.pt` holds model, optimiser, scheduler, GradScaler, progress and RNG states (CPU RNG restored as a
CPU tensor); it is written atomically every `checkpoint_every_steps` and each epoch, and copied to `PERSIST_DIR`.
Selected inference weights (`model/weights.pt`) are kept separately.

In [ ]:
if TRAIN_MODEL:
    with log.stage("train / resume"):
        model, MODEL_CFG, TRAIN_STATE, TRAIN_SUBSET, SELECTION = P.train_model(
            ws, CFG, RUN_DIR, PERSIST_DIR, DEVICE, resume=RESUME, log=log)
    display(pd.DataFrame(TRAIN_STATE["history"]))
else:
    from huim.persist import load_model
    model, MODEL_CFG, _pre, _meta = load_model(Path(LOAD_RUN_DIR), DEVICE)
    assert _pre.to_json() == ws.pre.to_json()
    SELECTION, TRAIN_STATE, TRAIN_SUBSET = _meta["selection"], None, {"rule": "loaded; not retrained"}
    for rel in ("model/weights.pt", "model/model_config.json", "model/preprocessing.json", "model/article_ids.npy"):
        (RUN_DIR / rel).parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(Path(LOAD_RUN_DIR) / rel, RUN_DIR / rel)
    log(f"loaded model from {LOAD_RUN_DIR} (no fitting)")
PARAMS = parameter_counts(model)
print(SELECTION, PARAMS)

In [ ]:
# Optional bounded interruption/resume check (prepared; result only exists once executed).
INTERRUPTION_CHECK = None
if RUN_INTERRUPTION_CHECK and TRAIN_MODEL:
    from huim.train import Trainer, interruption_check, seed_everything
    from huim.model import build_model
    art_d, ctx_d = P.device_and_tensors(ws, DEVICE)
    # ~40k impressions (~110 steps at batch 4096) so the resume point (step 40) is genuinely mid-epoch
    subset, _ = P.training_subset(ws, dict(CFG, train_impression_limit=40_000))
    def build_trainer(folder):
        seed_everything(SEED)
        m = build_model(P.model_config(CFG, ws.pre, ws.art), ws.pre).to(DEVICE)
        return Trainer(m, art_d, ctx_d["train"], ctx_d["validation"], DEVICE,
                       dict(CFG, train_pair_indices=subset, checkpoint_every_steps=0), folder, None, log)
    with log.stage("interruption check"):
        INTERRUPTION_CHECK = interruption_check(build_trainer, RUN_DIR / "interruption_check", steps=40)
    atomic_write_json(RUN_DIR / "interruption_check.json", INTERRUPTION_CHECK)
    print(INTERRUPTION_CHECK)

In [ ]:
# Optional focused controls on RECORDED SLATES (same data, epochs, seed and optimiser; param counts recorded).
CONTROLS = None
if RUN_CONTROLS and TRAIN_MODEL:
    rows = []
    for name, override in {"uniform_pooling": {"attention_mode": "uniform"},
                           "gated_fusion": {"fusion": "gated"},
                           "unnormalised_attention": {"attention_mode": "unnormalised"}}.items():
        ccfg = dict(CFG, **override)
        cdir = RUN_DIR / "controls" / name
        with log.stage(f"control {name}"):
            cm, ccfg_model, cstate, _, csel = P.train_model(ws, ccfg, cdir, None, DEVICE, resume=RESUME, log=log)
        rows.append({"variant": name, **override, "best_epoch": cstate["best_epoch"],
                     "validation_auc": cstate["best_auc"],
                     "validation_ndcg@5": cstate["best_metrics"]["mean_impression_ndcg@5"],
                     "trainable_params": parameter_counts(cm)["total_trainable"]})
        del cm
    rows.append({"variant": "selected (main run)", "attention_mode": CFG["attention_mode"], "fusion": CFG["fusion"],
                 "best_epoch": SELECTION["selected_epoch"], "validation_auc": SELECTION["selected_validation_auc"],
                 "validation_ndcg@5": TRAIN_STATE["best_metrics"]["mean_impression_ndcg@5"],
                 "trainable_params": PARAMS["total_trainable"]})
    CONTROLS = pd.DataFrame(rows)
    CONTROLS.to_csv(RUN_DIR / "controls" / "controls.csv", index=False)
    display(CONTROLS)

In [ ]:
# Optional bounded Optuna search (validation AUC; development data). Does not replace the main run's
# selection automatically: copy chosen values into CONFIG_OVERRIDES and rerun, recording why.
STUDY_SUMMARY = None
if RUN_TUNING and TRAIN_MODEL:
    from huim.tune import run_study, SEARCH_SPACE
    def objective(trial, params):
        tcfg = dict(CFG, **params, max_epochs=TUNING_MAX_EPOCHS, train_impression_limit=TUNING_TRAIN_IMPRESSIONS,
                    checkpoint_every_steps=0)
        tdir = RUN_DIR / "tuning" / f"trial_{trial.number:03d}"
        _, _, tstate, _, _ = P.train_model(ws, tcfg, tdir, None, DEVICE, resume=True, log=log)
        return tstate["best_auc"]
    with log.stage("optuna"):
        study = run_study(objective, f"huim_{RUN_ID}", RUN_DIR / "tuning", PERSIST_DIR, TUNING_TRIALS, SEED)
    STUDY_SUMMARY = {"search_space": SEARCH_SPACE, "n_trials": len(study.trials), "best_params": study.best_params,
                     "best_value_validation_auc": study.best_value, "sampler": "TPE(seed)",
                     "budget": {"trials": TUNING_TRIALS, "epochs": TUNING_MAX_EPOCHS,
                                "train_impressions": TUNING_TRAIN_IMPRESSIONS}}
    atomic_write_json(RUN_DIR / "tuning" / "summary.json", STUDY_SUMMARY)
    display(study.trials_dataframe())

## 6. Validation diagnostics and explainability

Primary: validation mean impression AUC (complete composite-key slates, tie-fair, single-class impressions
excluded). Secondary: nDCG@5 (expected credit over ties). Then two focused diagnostics: (i) the same metrics with all
vectors hidden — relevant because most of the test catalogue has no vectors; (ii) attention weights for clicked vs
unclicked candidates and the mean attention entropy relative to uniform pooling. Attention is diagnostic, not proof of
reader motives or causality.

In [ ]:
import matplotlib.pyplot as plt
with log.stage("validation diagnostics"):
    VAL_METRICS, _ = P.evaluate_recorded(model, P.device_and_tensors(ws, DEVICE)[1]["validation"],
                                         P.device_and_tensors(ws, DEVICE)[0], DEVICE)
    VECTOR_MASKED = P.vector_masked_metrics(model, ws, DEVICE)
    ATTENTION = P.attention_examples(model, ws, DEVICE, n_requests=3, seed=SEED)
display(pd.DataFrame({"validation (development)": VAL_METRICS, "validation, vectors hidden": VECTOR_MASKED}))
print("mean attention entropy / log(n_valid):", ATTENTION["mean_entropy_ratio_vs_uniform"],
      "(1.0 = uniform; lower = more selective)")
ex = pd.DataFrame(ATTENTION["examples"])
display(ex.head(20))
atomic_write_json(RUN_DIR / "diagnostics" / "attention_examples.json", ATTENTION)
if len(ex):
    first = ex[ex.behaviour_id == ex.behaviour_id.iloc[0]]
    fig, ax = plt.subplots(figsize=(7, 3))
    for _, r in first.iterrows():
        ax.plot(range(1, len(r.top_weights) + 1), r.top_weights, marker="o",
                label=f"{r.candidate_article_id} ({'clicked' if r.clicked else 'not clicked'})")
    ax.set(xlabel="history item rank by weight", ylabel="attention weight", title=f"Request {first.behaviour_id.iloc[0]}")
    ax.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(RUN_DIR / "diagnostics" / "attention_example.png", dpi=120)
    plt.show()

## 7. Predict and export all splits

1. **Recorded-slate diagnostics** for all splits (`diagnostics/recorded_slates/{split}_scores.csv`), every original
   candidate exactly once, with catalogue-ineligible recorded candidates listed separately.
2. **Request-sampled development export** (`dev_sample/`, *not a submission*): whole requests, each with its **full
   eligible pool**; full scores + top-100, validated block by block; throughput measured for refined estimates.
3. **Contract exports** for `CFG["shortlist_splits"]`: top-100 for every request; full catalogue score CSVs only for
   splits that pass the storage gate. A split that fails the gate keeps complete shortlists and the manifest states that
   its full scores are absent and why; the pool is never silently changed.

In [ ]:
with log.stage("recorded-slate exports"):
    RECORDED, RECORDED_LOGITS = P.export_recorded(model, ws, RUN_DIR, DEVICE)
display(pd.DataFrame({s: {"rows": r["rows"], "valid": r["validation"]["passed"],
                          "ineligible_in_catalogue": r["catalogue_ineligible_recorded_candidates"],
                          **({k: r["metrics"][k] for k in ("mean_impression_auc", "mean_impression_ndcg@5")}
                             if "metrics" in r else {})} for s, r in RECORDED.items()}).T)

In [ ]:
DEV_ROOT = RUN_DIR / "dev_sample"
DEV_SETS = {s: C.sample_requests(len(ws.ctx[s].requests), CFG["dev_sample_requests"][s], SEED + i)
            for i, s in enumerate(SPLITS)}
DEV_ROOT.mkdir(parents=True, exist_ok=True)
(DEV_ROOT / "SAMPLE_NOT_A_SUBMISSION.md").write_text(
    "Request-sampled development export. Each sampled request keeps its full eligible catalogue pool.\n"
    "These files are NOT catalogue submissions and do not cover all requests.\n")
atomic_write_json(DEV_ROOT / "dev_sample_requests.json",
                  {**{s: v.tolist() for s, v in DEV_SETS.items()},
                   "rule": "uniform sample of whole requests without replacement", "seeds": {s: SEED + i for i, s in enumerate(SPLITS)},
                   "behaviour_ids": {s: ws.ctx[s].requests.behaviour_id.to_numpy()[v].tolist() for s, v in DEV_SETS.items()}})
with log.stage("dev-sample catalogue export"):
    DEV_RESULTS = P.export_catalogue(model, ws, CFG, DEV_ROOT, DEVICE, DEV_SETS, {s: True for s in SPLITS},
                                     RECORDED_LOGITS, log=log, check_blocks=None)
with log.stage("throughput benchmark"):
    art_d, ctx_d = P.device_and_tensors(ws, DEVICE)
    THROUGHPUT = {s: C.measure_throughput(C.CatalogueScorer(model, ctx_d[s], art_d, DEVICE, CFG["pair_chunk"],
                                                            CFG["inference_amp"]), DEV_SETS[s], CFG["request_block"])
                  for s in SPLITS}
REFINED = {}
for s in SPLITS:
    el = C.catalogue_eligibility(ws.ctx[s], ws.art)
    tp = THROUGHPUT[s]
    REFINED[s] = C.feasibility_estimate(el, ws.ctx[s], bytes_per_row=tp.get("csv_bytes_per_row"),
                                        pairs_per_second=tp.get("pairs_per_second_scoring_only"),
                                        csv_rows_per_second=tp.get("csv_rows_per_second"))
display(pd.DataFrame(REFINED).T[["requests", "eligible_pairs", "full_scores_csv_bytes_estimate",
                                 "scoring_hours_estimate", "csv_write_hours_estimate"]])
display(pd.DataFrame({s: {**r["summary"].get("recall", {}), "validation_passed": r["validation"]["passed"],
                          "recorded_vs_catalogue_max_diff": r["validation"]["checks"].get("recorded_vs_catalogue_max_abs_logit_diff")}
                      for s, r in DEV_RESULTS.items()}).T)

In [ ]:
FULL_RESULTS, GATE = {}, None
if CFG["shortlist_splits"]:
    GATE = P.full_score_gate({"eligibility": {s: {"feasibility": REFINED[s]} for s in SPLITS}}, CFG,
                             [LOCAL_ROOT, OUTPUT_ROOT], THROUGHPUT.get("test"))
    display(pd.DataFrame(GATE).T)
    sets = {s: np.arange(len(ws.ctx[s].requests)) for s in CFG["shortlist_splits"]}
    with log.stage("contract catalogue exports (all requests)"):
        FULL_RESULTS = P.export_catalogue(model, ws, CFG, RUN_DIR, DEVICE, sets,
                                          {s: GATE[s]["write"] for s in sets}, RECORDED_LOGITS, log=log,
                                          check_blocks=CFG["validation_check_blocks"])
    display(pd.DataFrame({s: {**r["summary"].get("recall", {}), "requests": r["summary"]["requests"],
                              "empty_pools": r["summary"]["empty_pools"],
                              "full_scores_written": r["summary"]["full_scores_written"],
                              "validation_passed": r["validation"]["passed"]} for s, r in FULL_RESULTS.items()}).T)
else:
    print("No contract catalogue exports in this mode (shortlist_splits is empty).")

In [ ]:
# ---- Manifests, metrics, run configuration, dependencies; then persist with a completion marker ----
def file_entry(path):
    return {"path": str(Path(path).relative_to(RUN_DIR)), "bytes": Path(path).stat().st_size} if Path(path).is_file() else None

contract = {}
for s in SPLITS:
    res = FULL_RESULTS.get(s)
    contract[s] = {
        "rankings": file_entry(RUN_DIR / "rankings" / s / "submission_top_100.csv"),
        "catalogue_scores": file_entry(RUN_DIR / "predictions" / s / "submission_click_probabilities.csv"),
        "catalogue_scores_status": ("written" if res and res["summary"]["full_scores_written"] else
                                    "not exported: " + ((GATE or {}).get(s, {}).get("reason") or
                                                        f"not part of RUN_MODE={RUN_MODE}")),
        "request_manifest": file_entry(RUN_DIR / "diagnostics" / "catalogue" / f"{s}_requests.csv"),
        "recorded_slate_scores": file_entry(RUN_DIR / "diagnostics" / "recorded_slates" / f"{s}_scores.csv"),
        "requests": int(len(ws.ctx[s].requests)),
        "requests_exported": res["summary"]["requests"] if res else 0,
        "eligible_pairs": int(PREFLIGHT["eligibility"][s]["feasibility"]["eligible_pairs"]),
        "empty_pools": res["summary"]["empty_pools"] if res else None,
        "validation": res["validation"] if res else None,
    }
CATALOGUE_VERSIONS = {name: INPUT_MANIFEST.get(f"articles.{name}") for name in ("train_val", "test")}
EXPORT_MANIFEST = {
    "export_schema_version": C.EXPORT_SCHEMA_VERSION, "prediction_pool": C.PREDICTION_POOL,
    "is_complete_submission": bool(RUN_MODE == "full" and all(contract[s]["catalogue_scores"] for s in SPLITS)
                                   and all(contract[s]["validation"] and contract[s]["validation"]["passed"] for s in SPLITS)),
    "run_mode": RUN_MODE, "eligibility_rules": C.ELIGIBILITY_RULES, "catalogue_policy": CFG["catalogue_policy"],
    "catalogue_versions": CATALOGUE_VERSIONS, "shortlist_size": C.SHORTLIST_SIZE,
    "final_display_size": C.FINAL_DISPLAY_SIZE, "score_semantics": C.SCORE_SEMANTICS,
    "score_file_name": "submission_click_probabilities.csv",
    "tie_breaker": "descending score, then ascending original article_id",
    "behaviour_id": "f'{user_id}:{impression_id}' from original integer IDs",
    "contract_exports": contract,
    "dev_sample": {"root": "dev_sample/", "is_submission": False, "requests": {s: len(v) for s, v in DEV_SETS.items()},
                   "results": {s: {"summary": r["summary"], "validation": r["validation"]} for s, r in DEV_RESULTS.items()}},
    "recorded_slates": {s: {k: v for k, v in r.items() if k != "metrics"} for s, r in RECORDED.items()},
    "feasibility": {"preflight": PREFLIGHT["eligibility"], "refined": REFINED, "throughput": THROUGHPUT, "gate": GATE},
}
atomic_write_json(RUN_DIR / "export_manifest.json", EXPORT_MANIFEST)

METRICS = {
    "definitions": P.METRIC_DEFINITIONS,
    "validation_development_recorded_slates": VAL_METRICS,
    "validation_recorded_slates_vectors_hidden": VECTOR_MASKED,
    "train_in_sample_recorded_slates": RECORDED["train"]["metrics"],
    "selection": SELECTION,
    "catalogue_known_click_recall": {
        **{f"{s}_all_requests": FULL_RESULTS[s]["summary"] for s in FULL_RESULTS if s != "test"},
        **{f"{s}_request_sample": DEV_RESULTS[s]["summary"] for s in DEV_RESULTS if s != "test"}},
    "test": "unavailable: official test split has no click labels",
    "attention_entropy_ratio": ATTENTION["mean_entropy_ratio_vs_uniform"],
    "controls": None if CONTROLS is None else CONTROLS.to_dict(orient="records"),
    "tuning": STUDY_SUMMARY, "interruption_check": INTERRUPTION_CHECK,
    "notes": ["Validation was used for early stopping/selection: development results, not an untouched test.",
              "Train metrics are in-sample diagnostics.",
              "Recall@100 measures recovery of observed clicks, not unbiased catalogue relevance or CTR."],
}
atomic_write_json(RUN_DIR / "metrics.json", METRICS)
RUN_CONFIG = {
    "experiment_id": P.EXPERIMENT_ID, "run_id": RUN_ID, "owner": OWNER, "seed": SEED, "run_mode": RUN_MODE,
    "train_model": TRAIN_MODEL, "loaded_from": str(LOAD_RUN_DIR) if LOAD_RUN_DIR else None,
    "code": {**CODE_REVISION, "helper_sha256": HELPER_HASHES, "repo_url": REPO_URL},
    "environment": environment_info(), "config": CFG, "storage_probes": STORAGE_PROBES,
    "data_root": str(DATA_ROOT), "resolved_paths": ws.paths.as_dict(), "input_manifest": "input_manifest.json",
    "feature_case": CFG["feature_case"], "feature_order": FEATURE_ORDER, "feature_audit": F.FEATURE_AUDIT,
    "missing_value_policy": MISSING_POLICY, "excluded_inputs": {**D.EXCLUDED_ARTICLE_COLUMNS, **D.EXCLUDED_BEHAVIOUR_COLUMNS},
    "preprocessing": "model/preprocessing.json", "model_config": MODEL_CFG.to_json(), "parameters": PARAMS,
    "training_subset": TRAIN_SUBSET, "selection": SELECTION, "embedding_sources": {
        "text": "gemini-embedding-2, 768-d, frozen lookup (Preprocessing/embedding_pipeline.py)",
        "image": "EB-NeRD supplied, 1024-d, frozen lookup", "combination": f"first source wins: {CFG['vector_source_order']}",
        "metadata": {k: str(ws.paths.vector_metadata.get(k)) for k in ws.paths.vector_metadata}},
    "history": {"source": "shared fixed history.parquet per split", "cutoff": "events strictly before each split's first request",
                "augmentation": "none (later behaviour clicks are not appended)", "items": CFG["history_len"],
                "reports": ws.reports["history"]},
    "coverage": PREFLIGHT["coverage"], "export_schema_version": C.EXPORT_SCHEMA_VERSION,
    "prediction_pool": C.PREDICTION_POOL, "eligibility_rules": C.ELIGIBILITY_RULES,
    "catalogue_versions": CATALOGUE_VERSIONS, "shortlist_size": C.SHORTLIST_SIZE, "score_semantics": C.SCORE_SEMANTICS,
    "export_manifest": "export_manifest.json", "tuning_objective": "validation mean impression AUC (maximise)",
    "persistent_location": str(PERSIST_DIR),
}
atomic_write_json(RUN_DIR / "run_config.json", RUN_CONFIG)
(RUN_DIR / "requirements.txt").write_text(P.requirements_text())

with log.stage("persist run bundle (copy + verify; marker comes after the reload check)"):
    PERSISTED = P.persist_run(RUN_DIR, PERSIST_DIR, log=log)
print("Persisted (not yet marked complete):", PERSIST_DIR)

## 8. Reload check

Loads weights, configuration and preprocessing **from the persistent copy** (no fitting), re-scores validation
recorded candidates and the first dev-sample catalogue requests, and compares with the persisted exports (tolerance
1e-4 on logits). Only if this passes is `_COMPLETE_RUN.json` written — last — so an unverified bundle is never treated
as complete. The result is only evidence once this cell has run.

In [ ]:
with log.stage("reload check from persistent copy"):
    RELOAD = P.reload_check(PERSIST_DIR, ws, CFG, DEVICE)
atomic_write_json(RUN_DIR / "reload_check.json", RELOAD)
shutil.copyfile(RUN_DIR / "reload_check.json", PERSIST_DIR / "reload_check.json")
print(json.dumps(RELOAD, indent=1))
assert RELOAD["passed"], "reload check failed — see reload_check.json; the run is NOT marked complete"
P.mark_run_complete(RUN_DIR, PERSIST_DIR, RELOAD, log=log)
print("complete:", P.run_is_complete(PERSIST_DIR))

## 9. Business interpretation and handoff

Fill this in **from the executed outputs above**; do not carry numbers from elsewhere.

* Validation (development) AUC / nDCG@5 and, if run, the uniform-pooling and gated-fusion controls: does candidate-
  conditioned matching help at a comparable budget? If not, the simpler control is sufficient.
* Catalogue known-click Recall@100 (all validation requests in `development`/`full`), eligibility exclusions, and the
  shortlist's vector coverage / median age / category spread — inputs to Stage 2's freshness and diversity work.
* Vector-hidden robustness: how much does ranking degrade without text/image vectors (most test-catalogue articles)?
* Limitations: exposure bias (trained only on shown slates), unknown outcomes outside recorded slates, catalogue
  snapshot differences between `train_val` and `test`, offline clicks ≠ retention.

Handoff for the Stage 2 owner: the persisted bundle at `PERSIST_DIR` (rankings, catalogue scores where written,
recorded-slate diagnostics, request manifests, model bundle). The cell below assembles the small review bundle.

In [ ]:
# ---- Small review bundle (no full data, no model binaries) --------------------------------------
import zipfile
REVIEW = RUN_DIR / f"review_bundle_{RUN_ID}.zip"
small = ["run_config.json", "metrics.json", "export_manifest.json", "preflight.json", "input_manifest.json",
         "training_history.json", "reload_check.json", "interruption_check.json", "requirements.txt",
         "logs/run.log", "logs/errors.log", "controls/controls.csv", "tuning/trials.csv", "tuning/summary.json",
         "diagnostics/attention_examples.json", "diagnostics/attention_example.png",
         "dev_sample/dev_sample_requests.json"]
small += [str(p.relative_to(RUN_DIR)) for p in RUN_DIR.glob("diagnostics/catalogue/*_export_progress.json")]
small += [str(p.relative_to(RUN_DIR)) for p in RUN_DIR.glob("dev_sample/diagnostics/catalogue/*_requests.csv")]
small += [str(p.relative_to(RUN_DIR)) for p in RUN_DIR.glob("diagnostics/recorded_slates/*_catalogue_ineligible.csv")]
with zipfile.ZipFile(REVIEW, "w", zipfile.ZIP_DEFLATED) as zf:
    for rel in small:
        if (RUN_DIR / rel).is_file() and (RUN_DIR / rel).stat().st_size < 200 * 1024 ** 2:
            zf.write(RUN_DIR / rel, rel)
shutil.copyfile(REVIEW, PERSIST_DIR / REVIEW.name)
print(f"Review bundle: {PERSIST_DIR / REVIEW.name} ({REVIEW.stat().st_size / 1e6:.1f} MB)")
print("Bundle location for Stage 2:", PERSIST_DIR)
print({s: contract[s]["catalogue_scores_status"] for s in SPLITS})
if IN_COLAB:
    from google.colab import files
    files.download(str(REVIEW))